# 01 - Analisis Exploratorio de Datos (EDA)

**Proyecto:** Clustering de productos de maquillaje Cosnova (ESSENCE / CATRICE)
**Objetivo del proyecto:** planear la cantidad a pedir en cada compra semestral por canal de distribucion.

Este notebook responde cuatro preguntas antes de modelar:

1. Que hay realmente en los datos y que problemas de calidad traen.
2. Como se reparte la venta entre canales, marcas y categorias de producto.
3. Que forma tiene la venta en el tiempo: hay estacionalidad semestral?
4. Cuantos productos venden de forma continua y cuantos son esporadicos.

Sin esas respuestas no se puede elegir bien las variables del clustering.

---

In [ ]:
# Permite importar el paquete src desde la carpeta notebooks/
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config as cfg
from src import utils

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
utils.aplicar_estilo_graficas()

# En notebook queremos ver las figuras en linea, no solo guardarlas en disco
%matplotlib inline

print("Raiz del proyecto:", RAIZ)
print("Archivo fuente existe:", cfg.ARCHIVO_EXCEL.exists())

## 1. Estructura real del archivo fuente

El archivo entregado por el negocio tiene **6 hojas**, no 3. Solo tres son hojas
de canal con historico mensual; las otras son insumos previos que no entran al modelo.

Vale la pena mirarlas antes de asumir la estructura.

In [ ]:
import openpyxl

wb = openpyxl.load_workbook(cfg.ARCHIVO_EXCEL, read_only=True, data_only=True)
print("Hojas del archivo:", wb.sheetnames)
for hoja in wb.sheetnames:
    ws = wb[hoja]
    uso = "SI se usa en el modelo" if hoja in cfg.HOJAS_CANAL else "no se usa"
    print(f"  {hoja:<12} {ws.max_row:>7} filas x {ws.max_column:>3} columnas   ({uso})")
wb.close()

### Hallazgo: los encabezados de mes son fechas, no texto

Las columnas de mes vienen como `datetime` (`2024-01-01`), no como el texto `ene-24`.
`ingesta.py` las convierte a la etiqueta `ene-24` antes de cualquier calculo; si se
asumiera texto, el pipeline no encontraria ni una columna de venta.

In [ ]:
crudo = pd.read_excel(cfg.ARCHIVO_EXCEL, sheet_name="Exito", engine="openpyxl", nrows=5)
print("Tipo de los encabezados de mes:")
for columna in list(crudo.columns)[13:17]:
    print(f"  {columna!r}  ->  tipo {type(columna).__name__}")
crudo.iloc[:, :13]

## 2. Ingesta: unificacion de los 3 canales

`ingesta.ejecutar()` hace todo el trabajo de limpieza y deja el maestro listo.
Vale la pena leer los mensajes: reportan los problemas de calidad que encontro.

In [ ]:
from src import ingesta

maestro = ingesta.ejecutar()
maestro.head()

### Problemas de calidad detectados en la fuente

Tres cosas que la ingesta corrige o reporta, y que conviene tener presentes al
leer cualquier resultado del modelo:

1. **`Status` viene escrito de tres formas distintas** para el mismo concepto:
   `Ex artcile` (con typo), `ex article` y `Ex article`. Si se filtrara por texto
   exacto quedarian dentro decenas de articulos descontinuados. La ingesta compara
   sobre texto normalizado (minusculas, sin tildes).

2. **`EAN/UPC-Code` tiene un duplicado real**: el codigo `4059729583222` aparece
   en un material Catrice y en uno Essence. No es un error de carga que se pueda
   deduplicar sin perder informacion, son dos productos distintos. Por eso la
   clave del modelo es **`Material` + `Canal`**, no el EAN.

3. **`Categoria` es constante** (`Maquillaje` en las 1.815 filas). No aporta nada
   al clustering; la jerarquia informativa real arranca en `Subcategoria`.

In [ ]:
# Verificacion de los tres hallazgos sobre los datos crudos
crudo_completo = pd.read_excel(cfg.ARCHIVO_EXCEL, sheet_name="Exito", engine="openpyxl")

print("1) Valores de Status en la fuente:")
print(crudo_completo["Status"].value_counts().to_string())

print("\n2) Cardinalidad de las llaves candidatas (una hoja = un canal):")
print(f"   Filas: {len(crudo_completo)}")
print(f"   EAN/UPC-Code unicos: {crudo_completo['EAN/UPC-Code'].nunique()}   <- hay 1 duplicado")
print(f"   Material unicos:     {crudo_completo['Material '].nunique()}   <- clave valida")

print("\n3) Cardinalidad de la jerarquia de producto:")
for columna in ["Categoría", "Subcategoría", "Grupo", "Tono"]:
    print(f"   {columna:<15} {crudo_completo[columna].nunique():>3} valores distintos")

## 3. Panorama general del maestro

Con el maestro limpio ya se puede dimensionar el problema.

In [ ]:
utils.resumen_dataframe(maestro, "Maestro unificado")

meses = [m for m in cfg.MESES if m in maestro.columns]
print(f"\nRegistros producto x canal: {len(maestro):,}")
print(f"Productos unicos:           {maestro['Material'].nunique():,}")
print(f"Unidades en 30 meses:       {utils.formatear_miles(maestro[meses].to_numpy().sum())}")

## 4. Distribucion de la venta por dimension

Donde esta el volumen: canal, marca, subcategoria y grupo de producto.

In [ ]:
from src import eda

fig = eda.graficar_distribucion_por_dimension(maestro)
plt.show()

### Lectura

**El canal manda sobre todo lo demas.** Farmatodo concentra cerca del 78% de las
unidades; Exito y Prosalon se reparten el resto. Esto justifica la decision de
tratar cada producto-canal como un registro independiente: un SKU con rotacion
alta en Farmatodo puede ser marginal en Prosalon, y comprarle igual para los tres
canales seria un error de planeacion.

**ESSENCE pesa cerca de 3 veces mas que CATRICE** en unidades. Al ser una variable
del modelo, el clustering puede separar grupos por marca cuando eso coincide con
un patron de venta distinto.

In [ ]:
# Cifras exactas de participacion
total = maestro[meses].to_numpy().sum()
for dimension in ["Canal", "Marca", "Subcategoria"]:
    print(f"\nParticipacion por {dimension}:")
    serie = maestro.groupby(dimension)[meses].sum().sum(axis=1).sort_values(ascending=False)
    for nivel, unidades in serie.items():
        print(f"  {nivel:<14} {utils.formatear_miles(unidades):>12}  ({utils.porcentaje(unidades, total):>5}%)")

## 5. Evolucion mensual: existe el patron semestral?

Esta es la pregunta central del proyecto. Si la venta no tuviera diferencia entre
primer y segundo semestre, la variable `estacionalidad` no aportaria nada y la
proyeccion podria hacerse con un promedio simple.

In [ ]:
fig = eda.graficar_evolucion_mensual(maestro)
plt.show()

In [ ]:
# Comparacion directa S1 vs S2 en los dos anios completos
for anio in [2024, 2025]:
    s1 = maestro[cfg.SEMESTRES[f"S1_{anio}"]].to_numpy().sum()
    s2 = maestro[cfg.SEMESTRES[f"S2_{anio}"]].to_numpy().sum()
    print(f"{anio}:  S1 = {utils.formatear_miles(s1):>10}   S2 = {utils.formatear_miles(s2):>10}   "
          f"ratio S2/S1 = {s2 / s1:.3f}")

s1_2026 = maestro[cfg.SEMESTRES["S1_2026"]].to_numpy().sum()
print(f"2026:  S1 = {utils.formatear_miles(s1_2026):>10}   S2 = (por planear)")

### Lectura

**El segundo semestre vende mas que el primero, de forma consistente en ambos
anios completos.** El ratio S2/S1 esta claramente por encima de 1 en 2024 y 2025,
con diciembre como pico marcado.

Esto tiene dos consecuencias directas en el modelo:

1. La variable `estacionalidad` (ratio S1 vs S2) si aporta informacion y entra al
   clustering.
2. La proyeccion de S2_2026 **no puede ser un promedio simple del historico**:
   tiene que aplicar un factor de paso S1 a S2. Ese factor se calcula por cluster
   en `evaluation.calcular_factor_cluster`.

Tambien se ve una tendencia general de crecimiento: 2025 vende mas que 2024, y
S1_2026 ya supera a S1_2025.

## 6. Mapa de calor: patron temporal de los productos top

El grafico usa escala logaritmica en el color. Sin ella, los 2 o 3 productos mas
vendidos aplastan visualmente al resto y no se distingue ningun patron.

In [ ]:
fig = eda.graficar_heatmap_productos(maestro, top=50)
plt.show()

### Lectura

Se distinguen dos comportamientos muy distintos entre los top 50:

- **Bandas continuas** (color parejo de izquierda a derecha): productos de
  rotacion permanente. Son los mas facilmente planeables.
- **Bloques que arrancan a mitad del periodo**: lanzamientos recientes. Su
  historial corto es exactamente el problema que el clustering viene a resolver,
  porque no tienen suficiente pasado propio para proyectar.

## 7. Cobertura: venta continua vs. esporadica

`cobertura_meses` es el porcentaje de los 30 meses en que el producto vendio al
menos una unidad. Es la variable que mas directamente informa la prioridad de compra.

In [ ]:
from src import feature_engineering as fe

features = fe.ejecutar(maestro)
fig = eda.graficar_cobertura(features)
plt.show()

### Lectura

La distribucion es **bimodal**: hay un grupo grande de productos que vende casi
todos los meses y otro grupo grande que casi no vende. Muy pocos productos estan
en el medio.

Esa forma es una buena senal para el clustering: significa que existen grupos
naturalmente distintos y que no estamos partiendo arbitrariamente una nube
homogenea.

El grupo de cobertura 0% (productos sin una sola venta en 30 meses) se maneja
aparte: no entra al clustering porque todas sus variables valen cero.

## 8. Dispersion del volumen por subcategoria y canal

Eje logaritmico, por el mismo motivo que el mapa de calor: el volumen abarca
varios ordenes de magnitud.

In [ ]:
fig = eda.graficar_boxplots(features)
plt.show()

## 9. Productos en crecimiento vs. en declive

`tendencia` compara la venta promedio mensual de S2_2025 contra la de S1_2024.
Positiva significa que el producto crece.

In [ ]:
fig = eda.graficar_tendencias(features)
plt.show()

## 10. Correlacion entre las variables numericas

Se usa correlacion de **Spearman** y no Pearson: las distribuciones de volumen
son muy sesgadas y Pearson quedaria dominado por los productos extremos.

El objetivo es detectar redundancia. Si dos variables correlacionan casi 1,
aportan la misma informacion.

In [ ]:
fig = eda.graficar_correlaciones(features)
plt.show()

### Lectura

`total_unidades`, `promedio_mensual` e `ingreso_estimado` estan fuertemente
correlacionadas, lo cual es esperable: las tres miden volumen.

Dos decisiones que salen de aqui:

- `ingreso_estimado` **no entra al clustering**: es basicamente `total_unidades`
  multiplicado por el precio, no aporta un eje nuevo. Se conserva para el reporte
  de negocio, donde si es la cifra que importa.
- `total_unidades` y `promedio_mensual` si entran ambas, pero el **PCA se encarga
  de colapsar lo que comparten**. Por eso el pipeline aplica PCA antes de
  agrupar: evita contar el volumen tres veces en la distancia.

`cobertura_meses`, `tendencia`, `estacionalidad` y `variabilidad` correlacionan
poco entre si: cada una aporta un eje distinto de comportamiento. Es justo lo que
se busca en un set de features.

## 11. Concentracion de la venta y resumen de hallazgos

In [ ]:
eda.resumen_eda(maestro, features)

---

## Conclusiones del EDA

**Sobre la calidad de los datos**
1. `Status` requiere normalizacion antes de filtrar: tres escrituras del mismo valor.
2. La clave del modelo es `Material` + `Canal`; el EAN tiene un duplicado real.
3. `Categoria` es constante y se descarta; la jerarquia util empieza en `Subcategoria`.

**Sobre el negocio**
4. Farmatodo concentra cerca del 78% del volumen: el canal es un factor de primer orden.
5. El segundo semestre vende mas que el primero de forma consistente. La proyeccion
   necesita un factor de paso S1 a S2, no un promedio simple.
6. La venta esta muy concentrada: alrededor del 16% de los registros hace el 80%
   de las unidades.
7. Una parte importante del portafolio no vendio nada en 30 meses. Se separa del
   clustering y se marca para decision comercial.

**Sobre las variables del modelo**
8. La cobertura es bimodal: hay grupos naturalmente separados.
9. Las variables de volumen estan correlacionadas entre si; el PCA resuelve la
   redundancia. `ingreso_estimado` queda fuera del modelo y solo se usa en reportes.
10. Cobertura, tendencia, estacionalidad y variabilidad aportan ejes independientes.

**Siguiente paso:** `02_Feature_Engineering.ipynb`, donde se construyen y validan
estas variables en detalle.